# Phase IV — Weather and MLP comparisons

The primary Phase III results remain the pre-existing study reference. The Phase IV fit window is August–October 2025. Each arm scores November and December with one frozen model and transformer. G4 and weighting comparisons are exploratory because those holdout months were previously examined. The MLP architecture is selected only on April–October development folds.

This notebook reads published aggregates. When a Phase IV aggregate is unavailable, its section reports that evidence is pending.

In [ ]:
from pathlib import Path
import csv
import json
from IPython.display import Markdown, display
import matplotlib.pyplot as plt

here = Path.cwd()
repo = next(p for p in (here, *here.parents) if (p / 'reports/phase3').is_dir())
phase3 = repo / 'reports/phase3'
phase4 = repo / 'reports/phase4'

def csv_rows(path):
    if path.exists():
        with path.open(encoding='utf-8', newline='') as file:
            return list(csv.DictReader(file))
    return []

def table(rows, columns):
    if rows:
        lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
        lines.extend('| ' + ' | '.join(str(r.get(c, '')).replace('|', '/') for c in columns) + ' |' for r in rows)
        display(Markdown('\n'.join(lines)))
    else:
        print('Verified aggregate evidence is pending.')

prior = csv_rows(phase3 / 'metrics_summary.csv')
historical = [r for r in prior if r['model'] == 'xgboost'
              and r['feature_view'] in {'G1_G2', 'G1_G2_G3'}
              and r['airport'] == 'overall' and 'pooled' in r['scored_month'].lower()]
table(historical, ['feature_view', 'scored_month', 'n', 'macro_f1', 'log_loss', 'severe_precision', 'severe_recall'])

## Architecture and paired models

Six unweighted candidates screen on April, July, and October. The top two are compared across all seven development folds by equal-month log loss. Inner validation selects epochs within each training window. The median stopping epoch of the selected candidate fixes the final training length.

XGBoost uses the Phase III G1–G3 settings for both fresh fits. MLP-A and MLP-B use G1–G3. MLP-C and MLP-D add G4. MLP-B and MLP-D use inverse-frequency class weighting.

In [ ]:
tuning = csv_rows(phase4 / 'mlp_tuning_summary.csv')
table(tuning, ['candidate_id', 'month', 'epochs', 'log_loss', 'macro_f1', 'stop_reason'])
seed_summary = csv_rows(phase4 / 'seed_summary.csv')
table([r for r in seed_summary if r['subgroup'] == 'overall'],
      ['arm', 'month', 'seed_count', 'macro_f1', 'log_loss', 'severe_precision', 'severe_recall',
       'severe_pr_auc_ap', 'actual_severe_prevalence', 'mean_predicted_severe_probability', 'argmax_severe_share'])

## Severe risk and paired contrasts

Calibration compares mean severe probability with observed severe frequency within each bin. The argmax severe share is the proportion classified as severe. The precision–recall curves are saved numerical aggregates. Factorial contrasts are raw model-minus-comparator differences, including for log loss.

In [ ]:
details_path = phase4 / 'metric_details.json'
details = json.loads(details_path.read_text(encoding='utf-8')) if details_path.exists() else {}
pooled = {key: values for key, values in details.items() if key.endswith('/pooled/overall')}
if pooled:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout='constrained')
    for key, values in sorted(pooled.items()):
        label = '/'.join(key.split('/')[:2])
        bins = [r for r in values['calibration'] if r['class_id'] == 2 and r['n']]
        axes[0].plot([r['mean_probability'] for r in bins], [r['observed_rate'] for r in bins], label=label)
        curve = values.get('severe_pr_curve', [])
        if curve:
            recall, precision = zip(*curve)
            axes[1].plot(recall, precision, label=label)
    axes[0].plot([0, 1], [0, 1], color='gray', linestyle=':')
    axes[0].set(xlabel='Mean severe probability', ylabel='Observed severe fraction')
    axes[1].set(xlabel='Severe recall', ylabel='Severe precision')
    axes[1].legend(fontsize='small')
    plt.show()
else:
    print('Verified severe-risk traces are pending.')

contrasts = csv_rows(phase4 / 'factorial_contrasts.csv')
table([r for r in contrasts if r['subgroup'] == 'overall' and r['month'] == 'pooled'
       and r['metric'] in {'macro_f1', 'log_loss', 'severe_recall', 'severe_precision'}],
      ['seed', 'metric', 'B_minus_A', 'C_minus_A', 'D_minus_C', 'D_minus_B', 'interaction'])

## Airports, lead time, weather and research questions

Lead-time bands use the interval from prediction to scheduled departure. The adverse band includes any MVFR-or-worse origin or destination, frozen precipitation, or thunderstorm. Benign requires those four signals to be observed and negative. The remaining cases are unknown.

These comparisons describe predictive performance. The evaluated period is within 2025 and correlated flight counts do not justify IID confidence intervals.

In [ ]:
table([r for r in seed_summary if r['subgroup'] != 'overall' and r['month'] == 'pooled'],
      ['arm', 'subgroup', 'seed_count', 'macro_f1', 'log_loss', 'severe_precision', 'severe_recall'])
for name in ('research_question_summary.md', 'PHASE4_VALIDATION.md'):
    path = phase4 / name
    if path.exists():
        display(Markdown(path.read_text(encoding='utf-8')))